# Qwen3-VL Hybrid - Kaggle
YOLO detects, Qwen3-VL explains each detection in one sentence.

**Fixes applied vs. the previous version**
- Correct Kaggle dataset paths (`/kaggle/input/<slug>/`), with auto-discovery as a safety net.
- `float16` on T4 GPUs (Turing has no bf16 acceleration → bf16 was the main cause of slowness). bf16 is used automatically on Ampere+.
- Highlighted image is down-scaled before it reaches the VLM, cutting vision tokens (the biggest speed lever) while keeping the red box visible.
- `max_new_tokens` lowered to 64 (one sentence), per-call VRAM cleanup added.
- `MIN_CONFIDENCE` filter added to match the rest of the project, and a `predictions.json` is now written for the downstream tools.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate ultralytics pillow supervision==0.27.0rc6

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HUGGINGFACE_TOKEN')
print('Token loaded.')

In [ ]:
# ── Config + robust path resolution ─────────────────────────────────
# Kaggle mounts each attached dataset at /kaggle/input/<dataset-slug>/ .
# Set the slugs below to match yours; if they are wrong the cell auto-discovers
# the YOLO weights and the image folder anywhere under /kaggle/input.
import glob
from pathlib import Path

IMAGES_DIR      = '/kaggle/input/rq2-explanation-sample-flat'
YOLO_MODEL_PATH = '/kaggle/input/yolo11s-visdrone-finetuned/fine-tuned.pt'

MODEL_ID           = 'Qwen/Qwen3-VL-2B-Instruct'
MIN_CONFIDENCE     = 0.40   # ignore low-confidence YOLO boxes
MAX_DETS_PER_IMAGE = 30     # cap boxes explained per image (0 = no limit)
MAX_IMAGE_SIDE     = 1280   # down-scale long side before the VLM (speed)
MAX_NEW_TOKENS     = 64     # one sentence
OUTPUT_DIR         = '/kaggle/working'

def _first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

# Auto-discover YOLO weights if the configured path is missing
if not Path(YOLO_MODEL_PATH).is_file():
    found = _first('/kaggle/input/**/fine-tuned.pt') or _first('/kaggle/input/**/*.pt')
    if found:
        print(f'YOLO weights not at configured path; using discovered: {found}')
        YOLO_MODEL_PATH = found

# Auto-discover the image folder if the configured one has no .jpg files
if not (Path(IMAGES_DIR).is_dir() and list(Path(IMAGES_DIR).glob('*.jpg'))):
    jpgs = glob.glob('/kaggle/input/**/*.jpg', recursive=True)
    if jpgs:
        IMAGES_DIR = str(Path(jpgs[0]).parent)
        print(f'Images not at configured path; using discovered: {IMAGES_DIR}')

n_imgs = len(list(Path(IMAGES_DIR).glob('*.jpg')))
print('Images dir exists :', Path(IMAGES_DIR).is_dir())
print('YOLO weights exist:', Path(YOLO_MODEL_PATH).is_file())
print('Images found      :', n_imgs)
assert n_imgs > 0,                       'No .jpg images found - attach the dataset / fix IMAGES_DIR.'
assert Path(YOLO_MODEL_PATH).is_file(),  'YOLO weights not found - attach the dataset / fix YOLO_MODEL_PATH.'

In [ ]:
# ── Load YOLO ──────────────────────────────────────────────
from ultralytics import YOLO
yolo = YOLO(YOLO_MODEL_PATH)
print('YOLO loaded. Classes:', yolo.names)

In [ ]:
# ── Load Qwen3-VL ──────────────────────────────────────────
# fp16 on T4 (Turing has no bf16 tensor cores - bf16 runs in a slow fallback);
# bf16 is used automatically on Ampere+ (A100 / L4 / etc.). No 4-bit quant:
# quantizing the vision tower produces washed-out / gray-looking inputs.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# torch.cuda.is_bf16_supported() returns True even on T4 - bf16 is 'supported'
# there but NOT hardware-accelerated (Turing has no bf16 tensor cores), so it
# runs a slow fallback. Gate on compute capability instead (Ampere = 8.0+).
_cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
DTYPE = torch.bfloat16 if _cc >= 8 else torch.float16
print(f'GPU compute capability {_cc}.x -> using dtype: {DTYPE}')

# max_pixels caps how many vision tokens each image becomes - the dominant
# cost for Qwen-VL on a T4. ~590k px keeps the red box clearly visible.
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=768 * 768)
model     = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,            # half precision - fp16 on T4 (see above)
    device_map={'': 0},          # pin to one GPU; never shard a 2B model across the 2 T4s
    attn_implementation='sdpa',  # fast attention on T4 (flash-attn needs Ampere)
).eval()
print('Loaded ->',
      'device:', next(model.parameters()).device,
      '| dtype:', next(model.parameters()).dtype,
      '| VRAM:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

In [ ]:
# ── Inference helpers ────────────────────────────────────────
import torch
from PIL import Image, ImageDraw


def draw_box(image_path: str, bbox: list, color: str = 'red', width: int = 3) -> Image.Image:
    """Draw the bounding box on the full-resolution image."""
    img  = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    x1, y1, x2, y2 = [int(v) for v in bbox]
    for off in range(width):
        draw.rectangle([x1 - off, y1 - off, x2 + off, y2 + off], outline=color)
    return img


def resize_for_vlm(img: Image.Image, max_side: int = MAX_IMAGE_SIDE) -> Image.Image:
    """Cap the long side so the VLM sees far fewer vision tokens (big speedup)."""
    w, h = img.size
    scale = max_side / max(w, h)
    if scale < 1.0:
        img = img.resize((round(w * scale), round(h * scale)), Image.LANCZOS)
    return img


def qwen_explain(image: Image.Image, class_name: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt = (
        f'This is an aerial drone image. '
        f'A {class_name} is highlighted with a red bounding box. '
        f'In one sentence, describe the {class_name} you see inside the red box.'
    )

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': resize_for_vlm(image)},
            {'type': 'text',  'text': prompt},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # NOTE: repetition_penalty / no_repeat_ngram_size were removed - as
            # CPU logits processors they scan the ~800 image tokens every decode
            # step, pegging one core and starving the GPU (~24s -> a few s/exp).
        )

    trimmed = [g[len(i):] for i, g in zip(inputs.input_ids, gen)]
    text    = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

    del inputs, gen, trimmed
    torch.cuda.empty_cache()
    return text


print('Helpers defined.')

In [ ]:
# ── Smoke test (1 image) ─────────────────────────────────────
test_img = sorted(Path(IMAGES_DIR).glob('*.jpg'))[0]
r = yolo.predict(str(test_img), conf=MIN_CONFIDENCE, save=False, verbose=False)
boxes = r[0].boxes
print(f'{test_img.name}: {len(boxes)} detections above conf {MIN_CONFIDENCE}')

if len(boxes):
    box         = sorted(boxes, key=lambda b: float(b.conf[0]), reverse=True)[0]
    class_name  = yolo.names[int(box.cls[0])]
    highlighted = draw_box(str(test_img), box.xyxy[0].tolist())
    explanation = qwen_explain(highlighted, class_name)
    print(f'Class: {class_name} | Conf: {float(box.conf[0]):.2f}')
    print(f'Explanation: {explanation}')
else:
    print('No detections above threshold on this image - try another or lower MIN_CONFIDENCE.')

print('Smoke test done.')

In [ ]:
# ── Full run ──────────────────────────────────────────────
import json, time

all_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))
cap = MAX_DETS_PER_IMAGE or 'no'
print(f'Running on {len(all_paths)} images, up to {cap} detections each (conf ≥ {MIN_CONFIDENCE}).\n')

results_all = {}
predictions = {}
timing_log  = {}
run_start   = time.perf_counter()

for i, image_path in enumerate(all_paths):
    img_start = time.perf_counter()
    print(f'[{i+1}/{len(all_paths)}] {image_path.name}', end=' ... ', flush=True)

    r     = yolo.predict(str(image_path), conf=MIN_CONFIDENCE, save=False, verbose=False)
    boxes = r[0].boxes

    if len(boxes) == 0:
        print('no detections')
        results_all[image_path.name] = []
        predictions[image_path.name] = []
        timing_log[image_path.name]  = {'total_s': 0, 'n': 0, 'per_explanation_s': []}
        continue

    sorted_boxes = sorted(boxes, key=lambda b: float(b.conf[0]), reverse=True)
    selected     = sorted_boxes if not MAX_DETS_PER_IMAGE else sorted_boxes[:MAX_DETS_PER_IMAGE]

    image_results = []
    image_preds   = []
    per_exp_times = []

    for box in selected:
        class_id    = int(box.cls[0])
        class_name  = yolo.names[class_id]
        confidence  = round(float(box.conf[0]), 4)
        bbox        = box.xyxy[0].tolist()
        highlighted = draw_box(str(image_path), bbox)

        t0          = time.perf_counter()
        explanation = qwen_explain(highlighted, class_name)
        per_exp_times.append(round(time.perf_counter() - t0, 2))

        image_results.append({
            'class_id':    class_id,
            'class_name':  class_name,
            'confidence':  confidence,
            'bbox':        bbox,
            'explanation': explanation,
        })
        image_preds.append({
            'class_id':   class_id,
            'class_name': class_name,
            'confidence': confidence,
            'bbox':       bbox,
        })

    img_elapsed = time.perf_counter() - img_start
    avg_exp     = sum(per_exp_times) / len(per_exp_times) if per_exp_times else 0

    results_all[image_path.name] = image_results
    predictions[image_path.name] = image_preds
    timing_log[image_path.name]  = {
        'total_s':           round(img_elapsed, 2),
        'n':                 len(image_results),
        'avg_exp_s':         round(avg_exp, 2),
        'per_explanation_s': per_exp_times,
    }

    elapsed = time.perf_counter() - run_start
    eta     = (elapsed / (i + 1)) * (len(all_paths) - i - 1)
    print(f'{len(image_results)} detections [{img_elapsed:.1f}s | avg {avg_exp:.1f}s/exp | ETA {eta/60:.0f}min]')

    if (i + 1) % 10 == 0:
        with open(f'{OUTPUT_DIR}/qwen3vl_hybrid_partial.json', 'w') as fout:
            json.dump(results_all, fout, indent=2)

total = time.perf_counter() - run_start
all_exp_times = [t for v in timing_log.values() for t in v.get('per_explanation_s', [])]
overall_avg   = sum(all_exp_times) / len(all_exp_times) if all_exp_times else 0
print(f'\nFinished {len(all_paths)} images in {total/60:.1f} min | overall avg {overall_avg:.1f}s/explanation')

In [ ]:
# ── Save  (download from the Output tab) ─────────────────────────
import json

with open(f'{OUTPUT_DIR}/qwen3vl_hybrid_explanations.json', 'w') as f:
    json.dump({
        'model':   MODEL_ID,
        'setting': 'hybrid',
        'config':  {
            'min_confidence':     MIN_CONFIDENCE,
            'max_dets_per_image': MAX_DETS_PER_IMAGE,
            'max_image_side':     MAX_IMAGE_SIDE,
        },
        'images':  results_all,
    }, f, indent=2)

with open(f'{OUTPUT_DIR}/qwen3vl_hybrid_predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)

with open(f'{OUTPUT_DIR}/qwen3vl_hybrid_timing.json', 'w') as f:
    json.dump({'model': MODEL_ID, 'setting': 'hybrid', 'per_image': timing_log}, f, indent=2)

total_exp = sum(len(v) for v in results_all.values())
print(f'Saved {len(results_all)} images, {total_exp} explanations')
print('→ qwen3vl_hybrid_explanations.json')
print('→ qwen3vl_hybrid_predictions.json')
print('→ qwen3vl_hybrid_timing.json')